# **[DEPERECATED]**

# Пул кандидатов через единую скоринговую функцию

Раньше: каждый канал top-100 в union.\
 Потолок пула (recall@N без обрезки) = 0.5365 на 6 каналах - item, слабый по отдельности на каждом канале, не попадает в пул вообще.

Здесь: широкий пул из BM25 + гео, затем для ВСЕХ кандидатов в этом пуле считается один совокупный скор (BM25 + гео-бонус + категория-бонус + популярность-бонус), сорт по нему и top-500.

In [1]:
import sys

sys.path.append("..")

import numpy as np
import pandas as pd
import bm25s

from eval import DEFAULT_SEARCH_COLS, make_val_split, recall_at_50
from eval.metrics import recall_at_k
from geo import build_geo_tree, build_loc_centroid, haversine
from preprocessing import clean_text, extract_label_value, tokenize

train = pd.read_parquet("../data/train.parquet")
train_part, val_queries, val_relevant = make_val_split(train, val_size=0.1, random_state=42)
print("train_part:", len(train_part), "val_queries:", len(val_queries))

train_part: 448040 val_queries: 35447


Оценка на подвыборке val_queries (5000 из 35447) - полный проход с k=2000 по BM25 на всех был бы слишком долгим для одного прохода.

In [2]:
EVAL_SAMPLE = val_queries.sample(5000, random_state=1).reset_index(drop=True)
print("eval sample:", len(EVAL_SAMPLE))

eval sample: 5000


## Инфраструктура: BM25, гео, категории, популярность (train)

In [3]:
items_corpus = train.drop_duplicates("item_id")[
    ["item_id", "item_title_raw", "item_description_raw", "item_infm_params_text"]
].reset_index(drop=True)
corpus_item_ids = items_corpus["item_id"].tolist()
retriever = bm25s.BM25.load("../data/bm25s_index", load_corpus=False)

geo_tree, geo_item_ids = build_geo_tree(
    train.drop_duplicates("item_id")[["item_id", "item_location_id", "item_latitude", "item_longitude"]]
)
loc_centroid = build_loc_centroid(train_part)

item_meta = train.drop_duplicates("item_id").set_index("item_id")[[
    "item_location_id", "item_latitude", "item_longitude", "item_infm_params_text",
]].copy()
item_meta["item_latitude"] = item_meta["item_latitude"].astype(float)
item_meta["item_longitude"] = item_meta["item_longitude"].astype(float)

item_popularity = train_part["item_id"].value_counts()
max_pop = item_popularity.max()

item_vid = item_meta["item_infm_params_text"].apply(lambda t: extract_label_value(t, "Вид услуги"))
item_tip = item_meta["item_infm_params_text"].apply(lambda t: extract_label_value(t, "Тип услуги"))
print("инфраструктура готова:", len(items_corpus), "объявлений")

инфраструктура готова: 344825 объявлений


## Единый скор: BM25 (норм. по запросу) + гео-бонус + категория-бонус + популярность-бонус

In [4]:
BM25_K = 2000
GEO_K = 300
POOL_SIZE = 500

W_GEO, W_CAT, W_POP = 0.5, 0.3, 0.1 # на глаз


def score_pool_for_query(row, bm25_ids, bm25_scores, geo_ids, geo_dists):
    bm25_map = dict(zip(bm25_ids, bm25_scores))
    geo_map = dict(zip(geo_ids, geo_dists))
    pool = list(dict.fromkeys(list(bm25_ids) + list(geo_ids)))

    max_bm25 = max(bm25_scores) if len(bm25_scores) and max(bm25_scores) > 0 else 1.0

    q_vid = extract_label_value(row["search_infm_params_text"], "Вид услуги")
    q_tip = extract_label_value(row["search_infm_params_text"], "Тип услуги")

    scores = {}
    for iid in pool:
        bm25_norm = bm25_map.get(iid, 0.0) / max_bm25
        dist = geo_map.get(iid)
        geo_bonus = 1.0 / (1.0 + dist) if dist is not None else 0.0
        cat_bonus = 0.0
        if q_tip and item_tip.get(iid) == q_tip:
            cat_bonus = 1.0
        elif q_vid and item_vid.get(iid) == q_vid:
            cat_bonus = 0.5
        pop_bonus = np.log1p(item_popularity.get(iid, 0)) / np.log1p(max_pop)
        scores[iid] = bm25_norm + W_GEO * geo_bonus + W_CAT * cat_bonus + W_POP * pop_bonus
    ranked = sorted(scores.items(), key=lambda x: -x[1])
    return [iid for iid, _ in ranked[:POOL_SIZE]]


print("скоринговая функция готова")

скоринговая функция готова


## Прогон по сэмплу

In [5]:
BATCH = 500
pool_predictions = {}

for start in range(0, len(EVAL_SAMPLE), BATCH):
    batch = EVAL_SAMPLE.iloc[start:start + BATCH].reset_index(drop=True)
    q_tokens = batch["search_query"].apply(tokenize).tolist()
    bm25_res, bm25_sc = retriever.retrieve(q_tokens, corpus=corpus_item_ids, k=BM25_K, show_progress=False, n_threads=0)

    q_coords = loc_centroid.reindex(batch["search_location_id"])[["item_latitude", "item_longitude"]].to_numpy()
    valid = ~np.isnan(q_coords).any(axis=1)
    geo_idx_all = np.full((len(batch), GEO_K), -1, dtype=int)
    geo_dist_all = np.full((len(batch), GEO_K), np.nan)
    if valid.any():
        d, idx = geo_tree.query(np.radians(q_coords[valid]), k=GEO_K)
        geo_idx_all[valid] = idx
        geo_dist_all[valid] = d * 6371.0

    for i, row in batch.iterrows():
        g_ids = [geo_item_ids[j] for j in geo_idx_all[i] if j != -1]
        g_dists = [geo_dist_all[i][k] for k in range(len(geo_idx_all[i])) if geo_idx_all[i][k] != -1]
        pool_predictions[row["context_id"]] = score_pool_for_query(
            row, bm25_res[i], bm25_sc[i], g_ids, g_dists
        )

print("готово:", len(pool_predictions), "запросов обработано")

готово: 5000 запросов обработано


## Recall@50 (наивно по единому скору) и потолок пула (recall@500)

In [6]:
eval_relevant = {cid: val_relevant[cid] for cid in EVAL_SAMPLE["context_id"] if cid in val_relevant}

pool_recall_50 = recall_at_50({cid: p[:50] for cid, p in pool_predictions.items()}, eval_relevant)
pool_ceiling_500 = recall_at_k(pool_predictions, eval_relevant, k=POOL_SIZE)

print(f"Recall@50 (единый скор, без ранкера): {pool_recall_50:.4f}")
print(f"Потолок пула (recall@{POOL_SIZE}): {pool_ceiling_500:.4f}")
print("Прежний потолок (6 независимых каналов, top-100 каждый): 0.5365")
print("BM25 соло (для контекста, тот же тип сравнения): 0.1861")

Recall@50 (единый скор, без ранкера): 0.3332
Потолок пула (recall@500): 0.6489
Прежний потолок (6 независимых каналов, top-100 каждый): 0.5365
BM25 соло (для контекста, тот же тип сравнения): 0.1861


Веса ручками выбраны.
Дальнейший шаг - умные веса или замена на обученный скорер